## 1. 문서 가져오기

데이터 준비는 이 셀이 전부입니다. 별도의 데이터셋 라이브러리도, 스키마도 없습니다.

위키백과 API에 `explaintext=1` 옵션을 주면 위키 문법이 제거된 **순수 텍스트**를 받을 수 있습니다.
위키미디어는 요청에 `User-Agent` 헤더를 요구하므로 함께 보냅니다.

In [78]:
import re

import numpy as np
import requests
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
ai = OpenAI()

ENC = tiktoken.get_encoding("cl100k_base")
ntok = lambda t: len(ENC.encode(t))

In [79]:
TITLE = "운수 좋은 날"

resp = requests.get(
    "https://ko.wikisource.org/w/api.php",   # 위키백과가 아니라 위키문헌(원문 저장소)
    params={
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": 1,     # 위키 문법을 제거한 순수 텍스트로 받기
        "redirects": 1,
        "titles": TITLE,
    },
    headers={"User-Agent": "llm-edu-notebook/1.0"},
    timeout=30,
)

pages = resp.json()["query"]["pages"]
doc = next(iter(pages.values()))["extract"].strip()

# 내려받은 원문 끝에는 위키문헌이 붙인 '== 라이선스 ==' 절이 딸려 온다.
# 작품 내용이 아니므로 잘라낸다. (실무에서도 머리말·꼬리말 제거는 늘 필요한 단계)
doc = re.split(r"^== 라이선스 ==$", doc, flags=re.M)[0].strip()

print(f"{len(doc):,}자   {ntok(doc):,}토큰")
print(f"절(==) {len(re.findall(r'^== .+ ==$', doc, re.M))}개")
print(f"줄 {len(doc.splitlines())}개 / 빈 줄로 나눈 덩어리 {len(re.split(r'\n\s*\n', doc))}개")
print()
print(doc[:400])

10,284자   10,998토큰
절(==) 0개
줄 90개 / 빈 줄로 나눈 덩어리 4개

🙝🙟
새침하게 흐린 품이 눈이 올 듯하더니 눈은 아니 오고 얼다가 만 비가 추적추적 내리었다.
이날이야말로 동소문 안에서 인력거꾼 노릇을 하는 김 첨지에게는 오래간만에도 닥친 운수 좋은 날이었다. 문안에(거기도 문밖은 아니지만) 들어간답시는 앞집 마나님을 전찻길까지 모셔다 드린 것을 비롯으로 행여나 손님이 있을까 하고 정류장에서 어정어정하며 내리는 사람 하나하나에게 거의 비는 듯한 눈결을 보내고 있다가 마침내 교원인 듯한 양복장이를 동광학교(東光學校)까지 태워다 주기로 되었다.
첫번에 삼십 전, 둘째 번에 오십 전 - 아침 댓바람에 그리 흔치 않은 일이었다. 그야말로 재수가 옴붙어서 근 열흘 동안 돈 구경도 못한 김 첨지는 십 전짜리 백통화 서 푼, 또는 다섯 푼이 찰깍하고 손바닥에 떨어질 제 거의 눈물을 


## 2. 왜 잘라야 할까?

이유는 두 가지이고, 둘 다 구체적입니다.

1. **임베딩 모델에 입력 한계가 있습니다** — `text-embedding-3-small`은 8,192토큰까지만 받습니다.
   이 문서는 약 37,000토큰이라 **물리적으로 한 번에 넣을 수 없습니다.**
2. **들어간다고 해도 문서 전체를 벡터 하나로 만들면 쓸모가 없습니다.**
   임베딩은 텍스트 전체의 평균 같은 것입니다. 문서 전체를 벡터 하나로 만들면
   "인공지능 전반에 대한 것"이 되어 버려서, 어떤 질문에도 비슷하게 가깝고
   **어떤 질문에도 딱 맞지 않습니다.**

청킹은 문서를 **각 벡터가 한 가지 주제만 담을 만큼** 작게 나누는 작업입니다.

In [80]:
# 문서 전체를 임베딩하면 어떻게 될까? 직접 시도해서 거절당하는 것을 확인합니다.
try:
    ai.embeddings.create(model="text-embedding-3-small", input=[doc])
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:200])

BadRequestError
Error code: 400 - {'error': {'message': "Invalid 'input[0]': maximum input length is 8192 tokens.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


## 3. 자르는 방법 세 가지

아래 함수들은 모두 **텍스트를 받아 문자열 리스트를 돌려줄 뿐**입니다.
청커(chunker)란 그게 전부입니다.

| 방법 | 자르는 기준 | 특징 |
|---|---|---|
| `chunk_fixed` | 토큰 개수 | 일정한 크기. 문장 중간에서 잘림 |
| `chunk_paragraphs` | 빈 줄(문단) | 문장을 자르지 않음. 크기가 들쭉날쭉 |
| `chunk_sections` | `== 절 제목 ==` | 의미 단위로 완전. 너무 큼 |

In [81]:
def chunk_fixed(text, size=200, overlap=0):
    """`size` 토큰마다 무조건 자른다. 문장이 어디서 끝나는지는 신경 쓰지 않는다."""
    ids = ENC.encode(text)
    step = size - overlap
    out = []
    for i in range(0, len(ids), step):
        piece = ids[i : i + size]
        if not piece:
            break
        out.append(ENC.decode(piece))
        if i + size >= len(ids):
            break
    return out


def chunk_paragraphs(text, target=200):
    """문단을 통째로 모아서 대략 `target` 토큰이 될 때까지 합친다."""
    # \n+ : 빈 줄이든 한 줄 바꿈이든 문단으로 인정
    paras = [p.strip() for p in re.split(r"\n+", text) if p.strip()]
    out, buf, n = [], [], 0
    for p in paras:
        t = ntok(p)
        if buf and n + t > target:
            out.append("\n\n".join(buf))
            buf, n = [], 0
        buf.append(p)
        n += t
    if buf:
        out.append("\n\n".join(buf))
    return out


def chunk_sections(text):
    """'== 제목 ==' 절 단위로 자른다. 자연스럽지만 검색에 쓰기에는 너무 크다."""
    parts = re.split(r"(?=^== .+ ==$)", text, flags=re.M)
    return [p.strip() for p in parts if p.strip()]

In [82]:
for name, chunks in [
    ("fixed-200",       chunk_fixed(doc, 200)),
    ("fixed-200/ov50",  chunk_fixed(doc, 200, 50)),
    ("paragraphs-200",  chunk_paragraphs(doc, 200)),
    ("sections",        chunk_sections(doc)),
]:
    sizes = [ntok(c) for c in chunks]
    print(f"{name:18s} {len(chunks):>4}개 청크   "
          f"중앙값 {int(np.median(sizes)):>4}토큰   최대 {max(sizes):>5}토큰")

fixed-200            55개 청크   중앙값  200토큰   최대   200토큰
fixed-200/ov50       73개 청크   중앙값  200토큰   최대   200토큰
paragraphs-200       57개 청크   중앙값  190토큰   최대   345토큰
sections              1개 청크   중앙값 10998토큰   최대 10998토큰


## 4. 잘린 자리를 들여다보기

여기가 가장 눈여겨볼 부분입니다.
고정 크기 청킹은 토큰 수만 세다가 자르기 때문에, **문장 중간이나 심하면 단어 중간**에서 끊깁니다.

In [83]:
fixed = chunk_fixed(doc, 200)

print("=" * 72)
print("청크 5의 끝부분:")
print("=" * 72)
print("..." + fixed[5][-200:])
print()
print("=" * 72)
print("청크 6의 시작부분  <- 위 문장이 여기서 이어진다")
print("=" * 72)
print(fixed[6][:200] + "...")

청크 5의 끝부분:
...거지도록 누가 빼앗을 듯이 처박질 하더니만 그날 저녁부터 가슴이 땅긴다, 배가 켕긴다고 눈을 홉뜨고 지랄병을 하였다. 그때 김 첨지는 열화와 같이 성을 내며,
“에이, 오라질 년, 조롱복은 할 수가 없어, 못 먹어 병, 먹어서 병, 어쩌란 말이야! 왜 눈을 바루 뜨지 못해!”하고 김 첨지는 앓는 이의 뺨을 한 번 후려갈겼다. 홉

청크 6의 시작부분  <- 위 문장이 여기서 이어진다
뜬 눈은 조금 바루어졌건만 이슬이 맺히었다. 김 첨지의 눈시울도 뜨끈뜨끈하였다.
이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다.
“이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주...


In [84]:
# 문단 청킹은 줄 바꿈에서만 자르기 때문에 문장이 잘리지 않는다.
paras = chunk_paragraphs(doc, 200)

print("=" * 72)
print("문단 청크 5의 끝부분:")
print("=" * 72)
print("..." + paras[5][-200:])
print()
print("=" * 72)
print("문단 청크 6의 시작부분  <- 깔끔하게 나뉜다")
print("=" * 72)
print(paras[6][:200] + "...")

문단 청크 5의 끝부분:
...“에이, 오라질 년, 조롱복은 할 수가 없어, 못 먹어 병, 먹어서 병, 어쩌란 말이야! 왜 눈을 바루 뜨지 못해!”하고 김 첨지는 앓는 이의 뺨을 한 번 후려갈겼다. 홉뜬 눈은 조금 바루어졌건만 이슬이 맺히었다. 김 첨지의 눈시울도 뜨끈뜨끈하였다.

문단 청크 6의 시작부분  <- 깔끔하게 나뉜다
이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다.

“이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주는 마음이 시원치는 않았다....


### 겹침(overlap)은 무엇을 해 주나

겹침은 앞 청크의 끝부분을 다음 청크의 앞에 다시 넣습니다.
그래서 반으로 잘린 문장도 **어딘가에는 온전한 형태로 존재하게** 됩니다.
대가는 중복입니다. 청크가 늘어나고, 벡터가 늘어나고, 비용이 늘어납니다.

In [85]:
plain = chunk_fixed(doc, 200, 0)
over = chunk_fixed(doc, 200, 50)
print(f"겹침 없음 : {len(plain)}개 청크")
print(f"겹침 50   : {len(over)}개 청크  (저장·임베딩 비용 +{100 * (len(over) / len(plain) - 1):.0f}%)")
print()
print("청크 3의 마지막 50토큰이 청크 4의 앞부분에 다시 나타난다:")
print("---")
print(ENC.decode(ENC.encode(over[3])[-50:]))
print("---")
print(ENC.decode(ENC.encode(over[4])[:50]))

겹침 없음 : 55개 청크
겹침 50   : 73개 청크  (저장·임베딩 비용 +33%)

청크 3의 마지막 50토큰이 청크 4의 앞부분에 다시 나타난다:
---
 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈
---
 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈


## 5. 임베딩하고 검색하기

벡터 데이터베이스를 쓰지 않습니다.
청크가 수백 개뿐일 때는 **작은 행렬 하나와 내적(dot product) 한 번**이면 검색이 끝납니다.
이 규모에서 DB는 오히려 군더더기입니다.

벡터를 길이 1로 정규화해 두면 내적이 곧 **코사인 유사도**가 됩니다.

In [86]:
def embed(texts, batch=256):
    out = []
    for i in range(0, len(texts), batch):
        r = ai.embeddings.create(model="text-embedding-3-small",
                                 input=texts[i : i + batch])
        out.extend(d.embedding for d in r.data)
    V = np.asarray(out, dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True)   # 단위 벡터 -> 내적 == 코사인 유사도
    return V


INDEX = {}


def build(name, chunks):
    """한 번 임베딩한 결과를 기억해 둔다. (같은 이름은 다시 계산하지 않음)"""
    if name not in INDEX:
        print(f"임베딩 중: {name} ({len(chunks)}개 청크, 약 ${sum(map(ntok, chunks))/1e6*0.02:.4f})")
        INDEX[name] = (chunks, embed(chunks))
    return INDEX[name]


def search(query, name, k=3):
    chunks, V = INDEX[name]
    q = embed([query])[0]
    sims = V @ q
    top = np.argsort(-sims)[:k]
    return [(float(sims[i]), chunks[i]) for i in top]
    return [(float(sims[i]), chunks[i]) for i in top]

### `search()` 가 하는 일

위 `search()` 네 줄이 **RAG 의 R(검색)** 전부입니다.

```python
chunks, V = INDEX[name]      # ① 미리 임베딩해 둔 청크와 벡터 꺼내기
q = embed([query])[0]        # ② 질문도 같은 방식으로 벡터로
sims = V @ q                 # ③ 전부와 한 번에 비교
top = np.argsort(-sims)[:k]  # ④ 점수 높은 k개의 '번호' 고르기
```

| 단계 | 모양 | 하는 일 |
|---|---|---|
| ① | `V` = (186, 1536) | 청크 186개 × 벡터 1536차원 |
| ② | `q` = (1536,) | 질문 벡터 하나 |
| ③ | `sims` = (186,) | 청크마다 점수 하나씩 |
| ④ | `top` = (3,) | 가장 높은 3개의 **인덱스** |

③에서 `V @ q` 가 곧 코사인 유사도인 이유는 `embed()` 마지막 줄에서
`V /= np.linalg.norm(...)` 로 **길이 1짜리 벡터로 만들어 두었기** 때문입니다.
길이가 1이면 내적이 코사인과 같아집니다.
반복문 없이 **행렬 곱 한 번으로 186개를 동시에** 비교합니다.

---

### `top = np.argsort(-sims)[:k]` 뜯어보기

한 줄에 세 가지가 겹쳐 있습니다. 작은 예로 풀어 봅니다.

```
sims  = [0.31, 0.88, 0.45, 0.72, 0.12]     청크 5개의 점수
인덱스  =    0     1     2     3     4
```

**1. `argsort` — 값이 아니라 '번호'를 돌려준다**

이것이 핵심입니다. `sort` 와 헷갈리기 쉽습니다.

```
np.sort(-sims)[:3]     ->  [-0.88, -0.72, -0.45]    값만 나온다 → 어느 청크인지 모른다
np.argsort(-sims)[:3]  ->  [1, 3, 2]                번호가 나온다 → chunks[1] 로 꺼낼 수 있다
```

우리가 알아야 할 것은 "0.88점이 있다"가 아니라 **"1번 청크가 0.88점이다"** 입니다.
마지막 줄 `chunks[i]` 에서 그 번호로 본문을 꺼내야 하기 때문입니다.
그래서 `sort` 가 아니라 `argsort` 입니다.

**2. `-sims` — 음수를 붙여 내림차순으로 뒤집기**

`np.argsort` 는 **오름차순만** 지원합니다. `reverse=True` 같은 옵션이 없습니다.
그래서 부호를 뒤집어, "가장 작은 것"이 "원래 가장 컸던 것"이 되게 만듭니다.

```
sims   = [ 0.31,  0.88,  0.45,  0.72,  0.12]
-sims  = [-0.31, -0.88, -0.45, -0.72, -0.12]
                    ↑ 가장 작다 = 원래 가장 크다

np.argsort(-sims) = [1, 3, 2, 0, 4]
```

**3. `[:k]` — 앞에서 k개만**

```
[1, 3, 2, 0, 4]  →  [:3]  →  [1, 3, 2]

1위 인덱스 1 (0.88)   2위 인덱스 3 (0.72)   3위 인덱스 2 (0.45)
```

아래 셀에서 직접 확인할 수 있습니다.

---

### 알아 두면 좋은 두 가지

**`np.argsort(sims)[::-1]` 과 같지 않습니다**

같아 보이지만 **동점일 때 순서가 달라집니다.**

```
t = [0.5, 0.9, 0.5, 0.9, 0.1]        1·3번이 동점, 0·2번이 동점

np.argsort(-t)      = [1, 3, 0, 2, 4]     동점이면 작은 번호가 먼저
np.argsort(t)[::-1] = [3, 1, 2, 0, 4]     뒤집으면서 동점 순서도 뒤집힌다
```

검색에서 동점이 자주 나오지는 않지만, `k=1` 로 하나만 고를 때는 다른 청크가 뽑힐 수 있습니다.

**청크가 아주 많아지면 `argpartition` 이 빠릅니다**

`argsort` 는 **전부 정렬**합니다. 3개만 필요한데 186개를 모두 줄 세우는 셈입니다.
이 노트북 규모에서는 차이가 없지만, 청크가 수십만 개가 되면 달라집니다.

| 20만 개에서 상위 3개 | 시간 |
|---|---|
| `np.argsort(-x)[:3]` | 8.0 ms |
| `np.argpartition(-x, 3)[:3]` | **1.8 ms** |

`argpartition` 은 "상위 k개를 앞쪽으로 몰아놓기"만 하고 그 안의 순서는 보장하지 않습니다.
순위까지 필요하면 그 k개만 다시 정렬하면 됩니다.

> 실무 벡터 DB 가 내부에서 쓰는 방식입니다.
> `chunk_basic_ko2.ipynb` 부터는 Chroma 를 쓰므로 이 부분을 DB 가 대신해 줍니다.

In [87]:
build("fixed-200", chunk_fixed(doc, 200))
build("paragraphs-200", chunk_paragraphs(doc, 200))
print("\n준비된 인덱스:", list(INDEX))

임베딩 중: fixed-200 (55개 청크, 약 $0.0002)
임베딩 중: paragraphs-200 (57개 청크, 약 $0.0002)

준비된 인덱스: ['fixed-200', 'paragraphs-200']


## 6. 질문해 보기

점수는 참고만 하고, **돌아온 내용을 직접 읽어서** 쓸만한지 판단해 보세요.

In [88]:
def show(query, name, k=3, width=400):
    print("=" * 72)
    print(f"Q: {query}      [{name}]")
    print("=" * 72)
    for score, text in search(query, name, k):
        snippet = " ".join(text.split())[:width]
        print(f"\n[{score:.3f}] {snippet}...")


show("아내가 앓아누워 기침하는 대목", "paragraphs-200")

Q: 아내가 앓아누워 기침하는 대목      [paragraphs-200]

[0.457] 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다....

[0.391] 첫번에 삼십 전, 둘째 번에 오십 전 - 아침 댓바람에 그리 흔치 않은 일이었다. 그야말로 재수가 옴붙어서 근 열흘 동안 돈 구경도 못한 김 첨지는 십 전짜리 백통화 서 푼, 또는 다섯 푼이 찰깍하고 손바닥에 떨어질 제 거의 눈물을 흘릴 만큼 기뻤었다. 더구나 이날 이때에 이 팔십 전이라는 돈이 그에게 얼마나 유용한지 몰랐다. 컬컬한 목에 모주 한 잔도 적실 수 있거니와 그보다도 앓는 아내에게 설렁탕 한 그릇도 사다줄 수 있음이다....

[0.383] “여보게 돈 떨어졌네, 왜 돈을 막 끼얹나.” 이런 말을 하며 일변 돈을 줍는다. 김 첨지는 취한 중에도 돈의 거처를 살피는 듯이 눈을 크게 떠서 땅을 내려다보다가 불시에 제 하는 짓이 너무 더럽다는 듯이 고개를 소스라치자 더욱 성을 내며, “봐라 봐! 이 더러운 놈들아, 내가 돈이 없나, 다리뼉다구를 꺾어놓을 놈들 같으니.”하고 치삼의 주워주는 돈을 받아, “이 원수엣 돈! 이 육시를 할 돈!”하면서, 풀매질을 친다. 벽에 맞아 떨어진 돈은 다시 술 끓이는 양푼에 떨어지며 정당한 매를 맞는다는 듯이 쨍하고 울었다....


In [89]:
show("설렁탕을 사 왔는데 왜 먹지를 못하니", "paragraphs-200")

Q: 설렁탕을 사 왔는데 왜 먹지를 못하니      [paragraphs-200]

[0.645] “설렁탕을 사다놓았는데 왜 먹지를 못하니, 왜 먹지를 못하니... 괴상하게도 오늘은! 운수가 좋더니만... ”...

[0.526] 이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다. “이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주는 마음이 시원치는 않았다....

[0.443] 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다....


In [90]:
show("김 첨지의 아내는 어떤 상태인가?", "paragraphs-200")

Q: 김 첨지의 아내는 어떤 상태인가?      [paragraphs-200]

[0.583] 김 첨지의 눈은 벌써 개개 풀리기 시작하였다. 석쇠에 얹힌 떡 두 개를 숭덩숭덩 썰어서 볼을 불룩거리며 또 곱배기 두 잔을 부어라 하였다. 치삼은 의아한 듯이 김 첨지를 보며, “여보게 또 붓다니, 벌써 우리가 넉 잔씩 먹었네, 돈이 사십 전일세.”라고 주의시켰다....

[0.528] 치삼의 끄는 손을 뿌리치더니 김 첨지는 눈물이 글썽글썽한 눈으로 싱그레 웃는다. “죽기는 누가 죽어.”하고 득의가 양양. “죽기는 왜 죽어, 생때같이 살아만 있단다. 그 오라질 년이 밥을 죽이지. 인제 나한테 속았다.”하고 어린애 모양으로 손뼉을 치며 웃는다....

[0.513] 김 첨지는 취중에도 설렁탕을 사가지고 집에 다달았다. 집이라 해도 물론 셋집이요, 또 집 전체를 세든 게 아니라 안과 뚝떨어진 행랑방 한 간을 빌려 든 것인데 물을 길어대고 한 달에 일 원씩 내는 터이다. 만일 김 첨지가 주기를 띠지 않았던들 한 발을 대문에 들여놓았을 제 그곳을 지배하는 무시무시한 정적(靜寂) — 폭풍우가 지나간 뒤의 바다 같은 정적에 다리가 떨렸으리라....


## 7. 실험: 청크 크기가 결과를 바꾸는가?

같은 문서, 같은 질문, 같은 임베딩 모델입니다. **청크 크기만** 바뀝니다.
결과를 읽고, 어느 쪽을 LLM에게 컨텍스트로 넘기고 싶은지 직접 판단해 보세요.

작은 청크는 정확하지만 주변 맥락이 잘립니다.
큰 청크는 맥락이 충분하지만 정작 답이 관계없는 내용에 파묻힙니다.

In [91]:
QUESTION = "아내가 앓아누워 기침하는 대목"

for size in [100, 200, 400, 800]:
    name = f"para-{size}"
    build(name, chunk_paragraphs(doc, size))
    score, text = search(QUESTION, name, k=1)[0]
    snippet = " ".join(text.split())
    print("=" * 72)
    print(f"청크 크기 {size:>4}토큰  |  {len(INDEX[name][0]):>3}개 청크  |  "
          f"1위 점수 {score:.3f}  |  이 청크는 {ntok(text)}토큰")
    print("=" * 72)
    print(snippet[:600] + ("..." if len(snippet) > 600 else ""))
    print()

임베딩 중: para-100 (76개 청크, 약 $0.0002)
청크 크기  100토큰  |   76개 청크  |  1위 점수 0.457  |  이 청크는 277토큰
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다.

임베딩 중: para-200 (57개 청크, 약 $0.0002)
청크 크기  200토큰  |   57개 청크  |  1위 점수 0.457  |  이 청크는 277토큰
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다.

임베딩 중: para-400 (36개 청크, 약 $0.0002)
청크 크기  400토큰  |   36개 청크  |  1위 점수 0.457  |  이 청크는 277토큰
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 

### 방금 무슨 일이 일어났나 — 넘어가기 전에 읽어 주세요

| 청크 크기 | 1위 점수 | 돌아온 내용 |
|---|---|---|
| 100 | 0.506 | 딥러닝을 정의한 문단 (209토큰) |
| 200 | 0.506 | **위와 완전히 같은 청크** |
| 400 | 0.506 | **위와 완전히 같은 청크** |
| 800 | 0.414 | 탐색·논리·확률 이야기로 시작하는 764토큰 덩어리. 딥러닝 설명이 중간에 묻힘 |

**100, 200, 400이 왜 똑같을까요?**

`chunk_paragraphs`는 문단을 **쪼개지 않고 모으기만** 합니다.
`chunk_size`는 "이 크기를 목표로 자른다"가 아니라 **"이 크기를 넘지 않게 문단을 합친다"는 상한**입니다.
우리가 찾는 답은 209토큰짜리 문단 **하나**에 통째로 들어 있습니다.
그래서 상한을 100으로 낮춰도 그 문단은 쪼개지지 않고 그대로 남습니다.

> 여기서 얻을 교훈: **`chunk_size`를 줄였는데 결과가 그대로라면 청커가 무시하고 있는 것입니다.**
> 숫자만 바꾸지 말고, 실제로 몇 개의 청크가 만들어졌는지(108 → 99 → 81 → 54개)와
> 가져온 청크의 실제 크기를 함께 확인하세요.

**800에서는 왜 점수가 떨어졌을까요?**

여러 문단이 하나로 합쳐지면서 딥러닝 설명이 **관계없는 내용에 희석**되었습니다.
임베딩은 텍스트 전체의 평균에 가깝기 때문에, 다른 주제가 섞일수록 질문과의 유사도가 낮아집니다.
답이 청크 안에 들어 있긴 하지만 뒤쪽에 묻혀 있어서, LLM에게 넘길 컨텍스트로는 불리합니다.

**점수가 높다고 좋은 청크는 아닙니다.**

6번의 "딥러닝이란 무엇인가?" 결과에서 2위를 보세요.

```
[0.433] === 기계 학습과 딥 러닝 ===...
```

**제목만 있고 내용이 하나도 없는 청크**입니다.
딥러닝에 대한 질문에 "기계 학습과 딥 러닝"이라는 제목이 의미상 가까운 건 당연합니다.
점수는 높지만 정보는 0입니다. 점수만 보고 판단하면 안 되는 이유입니다.

**한글 위키백과 문서의 특징도 하나 보입니다.**

3번 표에서 `paragraphs-200`의 최대 청크가 **1,295토큰**이었습니다. 목표는 200이었는데도요.
문단 하나가 이미 200토큰을 넘으면 청커가 손을 대지 못하기 때문입니다.
실무에서는 보통 **문단으로 자른 뒤, 너무 큰 문단은 다시 쪼개는** 방식을 함께 씁니다.

### 고정 크기 vs 문단 경계

같은 질문, 같은 목표 크기입니다.
차이는 **문장 경계를 지키느냐 마느냐** 하나뿐입니다.

In [92]:
for name in ["fixed-200", "paragraphs-200"]:
    score, text = search(QUESTION, name, k=1)[0]
    print("=" * 72)
    print(f"{name}   점수 {score:.3f}")
    print("=" * 72)
    print(" ".join(text.split())[:500])
    print()

fixed-200   점수 0.403
�하고 손바닥에 떨어질 제 거의 눈물을 흘릴 만큼 기뻤었다. 더구나 이날 이때에 이 팔십 전이라는 돈이 그에게 얼마나 유용한지 몰랐다. 컬컬한 목에 모주 한 잔도 적실 수 있거니와 그보다도 앓는 아내에게 설렁탕 한 그릇도 사다줄 수 있음이다. 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니

paragraphs-200   점수 0.457
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다.



### 결과: 문단 경계를 지킨 쪽이 이겼습니다

| | 1위 점수 | 돌아온 내용 |
|---|---|---|
| `fixed-200` | 0.385 | `기계 학습과 딥 러닝 ===` 로 **시작**. 절 제목이 반토막 난 채로 들어왔고, 마지막도 "딥 러닝은 이 가운데 여러" 에서 끊김 |
| `paragraphs-200` | **0.506** | 딥러닝을 정의하는 문단이 처음부터 끝까지 온전히 |

같은 목표 크기인데 점수 차이가 큽니다.
고정 크기 청킹은 토큰만 세다가 **문장 중간, 심지어 제목 중간**에서 끊었고,
그렇게 잘린 조각은 의미가 온전하지 않아 질문과의 유사도도 낮게 나옵니다.

잘린 청크를 LLM에게 컨텍스트로 넘기면 답변 품질도 함께 떨어집니다.
**문장 경계를 지키는 것만으로 검색 품질이 올라갑니다.**

## 8. 직접 해 보기

질문을 바꿔 가며 다시 실행해 보세요. 시도해 볼 만한 것들입니다.

- **답이 여러 곳에 흩어진 질문** (*"인공지능의 역사는 어떻게 되는가?"*)
  — 청크 하나에 답이 다 들어 있나요?
- **문서에 없는 내용** (*"김치찌개 끓이는 법은?"*)
  — 유사도 점수를 보세요. 임베딩 검색은 **항상 무언가를 돌려줍니다.**
    관련 내용이 없다는 것을 알려 주는 유일한 단서가 점수입니다.
- **구체적인 용어**(*"퍼셉트론"*) vs **막연한 주제**(*"인공지능이 무섭다"*)
  — 어느 쪽을 임베딩 검색이 더 잘 처리하나요?

In [93]:
MY_QUESTION = "설렁탕을 사 왔는데 왜 먹지를 못하니"

show(MY_QUESTION, "paragraphs-200", k=3)

Q: 설렁탕을 사 왔는데 왜 먹지를 못하니      [paragraphs-200]

[0.645] “설렁탕을 사다놓았는데 왜 먹지를 못하니, 왜 먹지를 못하니... 괴상하게도 오늘은! 운수가 좋더니만... ”...

[0.526] 이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다. “이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주는 마음이 시원치는 않았다....

[0.443] 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다....


In [94]:
# 작품에 답이 없는 질문 -- 내용보다 점수를 보세요.
show("파이썬으로 웹 서버 만드는 법은?", "paragraphs-200", k=3)

Q: 파이썬으로 웹 서버 만드는 법은?      [paragraphs-200]

[0.166] “젠장맞을 것! 이 비를 맞으며 빈 인력거를 털털거리고 돌아를 간담. 이런 빌어먹을, 제 할미를 붙을 비가 왜 남의 상판을 딱딱 때려!”...

[0.117] 그는 불행에 다닥치기 전 시간을 얼마쯤이라도 늘리려고 버르적거렸다. 기적(奇蹟)에 가까운 벌이를 하였다는 기쁨을 할 수 있으면 오래 지니고 싶었다. 그는 두리번두리번 사면을 살피었다. 그 모양은 마치 자기 집 — 곧 불행을 향하고 달려가는 제 다리를 제 힘으로는 도저히 어찌할 수 없으니 누구든지 나를 좀 잡아 다고, 구해 다고 하는 듯하였다....

[0.116] 그는 몹시 홧증을 내며 누구에게 반항이나 하는 듯이 게걸거렸다. 그럴 즈음에 그의 머리엔 또 새로운 광명이 비쳤나니 그것은 <이러구 갈 게 아니라 이 근처를 빙빙 돌며 차 오기를 기다리면 또 손님을 태우게 될는지도 몰라>란 생각이었다. 오늘 운수가 괴상하게도 좋으니까 그런 요행이 또한번 없으리라고 누가 보증하랴. 꼬리를 굴리는 행운이 꼭 자기를 기다리고 있다고 내기를 해도 좋을 만한 믿음을 얻게 되었다. 그렇다고 정거장 인력거꾼의 등살이 무서우니 정거장 앞에 섰을 수는 없었다....


> **점수를 비교해 보세요.** 문서에 답이 있는 질문은 0.5~0.7이 나왔는데,
> 김치찌개 질문은 **0.294, 0.230, 0.216** 입니다.
> 그런데도 검색은 **빈손으로 돌아오지 않습니다.** 무조건 세 개를 가져옵니다.
>
> 실무에서는 이 점수에 **하한선(threshold)** 을 두어, 기준보다 낮으면
> "관련 문서를 찾지 못했습니다"라고 처리합니다.
> 그러지 않으면 LLM이 엉뚱한 청크를 근거로 그럴듯한 답을 지어냅니다.